# Reproducible 7-Model Communication-Function Pipeline

One notebook, one prompt, one column configuration, one export convention.

This notebook is designed for:
- sequential inference across **7 models**
- **4 GPU** runtime
- a **single shared prompt**
- a **single shared column block**
- a **20-row smoke test** before the full run
- full-corpus annotation
- per-model CSV exports named `output_dimnesions_scores_<model_slug>.csv`
- parse-rate, throughput, and convergence-style diagnostics after the run


## 1) Imports

In [ ]:
import os
import gc
import json
import math
import re
import shutil
import subprocess
import time
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from IPython.display import HTML, display
from vllm import LLM, SamplingParams


## 2) Hard-coded runtime parameters

In [ ]:
# =========================
# HARD-CODED GLOBAL PARAMETERS
# =========================

# 4 GPU setup
CUDA_VISIBLE_DEVICES = "0,1,2,3"
TENSOR_PARALLEL_SIZE = 4

# Project paths
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = Path("/home/cbenavent/test/Arcom_rares")

DATASET_PATH = PROJECT_ROOT / "data" / "annotation_working_master_human_2100_seed.csv"
OUTPUT_ROOT = PROJECT_ROOT / "communication_function_outputs" / "seven_model_reproducible_run"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Inference scope
SMOKE_TEST_ROWS = 20
FULL_RUN_LIMIT = None   # set to an integer if you want to debug on fewer than 8938 rows

# Prompt / parsing
ALLOW_DECIMALS = True
SCORE_MIN = 1.00
SCORE_MAX = 5.00
ROUND_DIGITS = 2

# Convergence-style proxy
CHUNK_SIZE = 500
CONVERGENCE_DRIFT_THRESHOLD = 0.08
CONVERGENCE_MIN_PARSE_RATE = 0.95

# Environment
os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
os.environ["VLLM_ENABLE_V1_MULTIPROCESSING"] = "1"
os.environ["MKL_THREADING_LAYER"] = "GNU"
os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"

conda_prefix = os.environ.get("CONDA_PREFIX", "")
if conda_prefix:
    conda_lib = f"{conda_prefix}/lib"
    ld_library_path = os.environ.get("LD_LIBRARY_PATH", "")
    ld_parts = [p for p in ld_library_path.split(":") if p]
    if conda_lib not in ld_parts:
        os.environ["LD_LIBRARY_PATH"] = f"{conda_lib}:{ld_library_path}" if ld_library_path else conda_lib

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATASET_PATH:", DATASET_PATH)
print("OUTPUT_ROOT:", OUTPUT_ROOT)
print("CUDA_VISIBLE_DEVICES:", os.environ.get("CUDA_VISIBLE_DEVICES"))
print("TENSOR_PARALLEL_SIZE:", TENSOR_PARALLEL_SIZE)


## 3) Model list

Edit only this block if you want to change the 7-model benchmark.

In [ ]:
# =========================
# CHANGE MODELS HERE IF NEEDED
# =========================

MODEL_SPECS = [
    {
        "label": "Gemma 4 26B",
        "model_name": "google/gemma-4-26B-A4B",
        "chat_mode": "manual_gemma",
        "dtype": "bfloat16",
        "gpu_memory_utilization": 0.88,
        "max_model_len": 6144,
        "max_new_tokens": 220,
        "disable_custom_all_reduce": True,
    },
    {
        "label": "Mistral Small 24B",
        "model_name": "mistralai/Mistral-Small-3.1-24B-Instruct-2503",
        "chat_mode": "hf_auto",
        "dtype": "bfloat16",
        "gpu_memory_utilization": 0.88,
        "max_model_len": 6144,
        "max_new_tokens": 220,
        "disable_custom_all_reduce": False,
    },
    {
        "label": "Mixtral 8x7B",
        "model_name": "mistralai/Mixtral-8x7B-Instruct-v0.1",
        "chat_mode": "hf_auto",
        "dtype": "bfloat16",
        "gpu_memory_utilization": 0.88,
        "max_model_len": 6144,
        "max_new_tokens": 220,
        "disable_custom_all_reduce": False,
    },
    {
        "label": "Llama 3.3 70B",
        "model_name": "meta-llama/Llama-3.3-70B-Instruct",
        "chat_mode": "hf_auto",
        "dtype": "bfloat16",
        "gpu_memory_utilization": 0.88,
        "max_model_len": 6144,
        "max_new_tokens": 180,
        "disable_custom_all_reduce": False,
    },
    {
        "label": "Qwen 2.5 14B",
        "model_name": "Qwen/Qwen2.5-14B-Instruct",
        "chat_mode": "hf_auto",
        "dtype": "bfloat16",
        "gpu_memory_utilization": 0.88,
        "max_model_len": 6144,
        "max_new_tokens": 220,
        "disable_custom_all_reduce": False,
    },
    {
        "label": "Qwen 2.5 32B",
        "model_name": "Qwen/Qwen2.5-32B-Instruct",
        "chat_mode": "hf_auto",
        "dtype": "bfloat16",
        "gpu_memory_utilization": 0.88,
        "max_model_len": 6144,
        "max_new_tokens": 220,
        "disable_custom_all_reduce": False,
    },
    {
        "label": "Qwen 2.5 72B",
        "model_name": "Qwen/Qwen2.5-72B-Instruct",
        "chat_mode": "hf_auto",
        "dtype": "bfloat16",
        "gpu_memory_utilization": 0.88,
        "max_model_len": 6144,
        "max_new_tokens": 220,
        "disable_custom_all_reduce": False,
    },
]

pd.DataFrame(MODEL_SPECS)[["label", "model_name", "chat_mode", "max_model_len", "max_new_tokens"]]


## 4) Column configuration

This is the block you should edit if you want more or fewer text columns.

In [ ]:
# =========================================================
# CHANGE OR ADD COLUMN NAMES HERE IF YOU WANT MORE OR LESS
# =========================================================

SELECTED_COLUMNS = [
    "Script",
    "Titre",
    "Visuel",
]

# Example:
# SELECTED_COLUMNS = ["Script", "Incrustation", "Titre", "Visuel"]

ROW_ID_COL = "row_id"

print("Columns currently used for model input:")
for col in SELECTED_COLUMNS:
    print("-", col)


## 5) Shared prompt

In [ ]:
RUBRIC_PROMPT = r"""You are an expert annotation assistant for French automotive advertising.

Your task is to analyse one ad and score it on three communication dimensions:
- informativeness
- expressiveness
- phatic

You must annotate carefully, conservatively, and consistently.
Your goal is not to be creative.
Your goal is to produce the most defensible annotation possible from the evidence in the ad.

You will receive ad text that may combine:
- script
- on-screen text
- title
- visual description

These elements may contain both literal information and symbolic or rhetorical cues.
You must judge the ad as a whole.

==================================================
TASK
==================================================

Score the ad on each dimension from 1.00 to 5.00.

- 1.00 = almost absent
- 2.00 = weak
- 3.00 = moderate
- 4.00 = strong
- 5.00 = very strong

Intermediate decimal values are allowed.
Always return values rounded to two decimals.

The three dimensions are independent.
An ad can be high on more than one dimension.
Do not force the scores to sum to any fixed total.

After scoring, also provide:
- dominant_dimension
- dominant_dimension_score
- confidence
- reason

If the highest score is shared by more than one dimension, dominant_dimension must be "mixed".
dominant_dimension_score must equal the highest score after rounding.

==================================================
DIMENSION DEFINITIONS
==================================================

A. INFORMATIVENESS

Definition:
How much the ad provides factual, concrete, product-related, offer-related, or technically useful information.

This includes:
- vehicle specifications
- features and equipment
- engine or powertrain information
- electric or hybrid technology
- charging, range, battery, consumption
- safety systems
- comfort or space features when presented concretely
- maintenance, guarantee, reliability claims when concrete
- price
- discounts
- financing, leasing, monthly payments
- trade-in conditions
- bonus or subsidy information
- model names, versions, product details
- explicit comparative or functional claims

High informativeness means:
the ad gives the viewer usable product or offer information.

Important rule:
Price, financing, technical details, and offer conditions are informative even if the ad is also emotional.

B. EXPRESSIVENESS

Definition:
How much the ad relies on emotion, desire, identity, aspiration, style, symbolic value, atmosphere, prestige, seduction, or aesthetic projection.

This includes:
- emotional appeal
- beauty, elegance, sensuality
- pleasure, passion, freedom, adventure
- self-image and identity
- desire and dream
- luxury and prestige
- symbolic staging
- strong aestheticization
- dramatic mood
- poetic or evocative language
- brand mythology
- visual spectacle used to create attraction rather than explain the product

High expressiveness means:
the ad primarily tries to make the car or brand desirable, meaningful, aspirational, stylish, moving, or emotionally charged.

C. PHATIC

Definition:
How much the ad creates, maintains, or foregrounds social contact, relational connection, conversational closeness, complicity, or audience bonding.

This includes:
- direct address to the viewer
- rhetorical interaction
- social or conversational tone
- familiar, intimate, complicit language
- greetings, invitations, banter, playful contact
- communication whose main role is to establish or maintain connection rather than inform or aesthetically seduce
- emphasis on interpersonal exchange, contact, or togetherness
- “we are talking to you” energy
- casual rapport-building discourse

High phatic means:
the ad is strongly oriented toward establishing or maintaining a relationship with the audience or between people.

==================================================
KEY DISTINCTIONS
==================================================

1. Informativeness vs Expressiveness
- Informativeness = concrete useful product or offer content
- Expressiveness = emotional or aesthetic persuasion

2. Expressiveness vs Phatic
- Expressiveness = emotion, aspiration, style, symbolism, mood
- Phatic = contact, social bond, conversational connection, relational presence

3. Informativeness vs Phatic
- Informativeness gives concrete content
- Phatic creates contact

==================================================
VERY IMPORTANT SCORING RULES
==================================================

Use the full scale carefully.
Do not inflate all dimensions.
Do not reward every polished ad with high expressiveness.
Do not reward every second-person or viewer-facing phrase with high phatic.
Do not ignore concrete offer or product information.

Do not score informativeness as 4.00 or 5.00 merely because one financing line, one price mention, or one technical detail appears. Use 4.00 or 5.00 only when factual or offer information is one of the ad’s major communicative forces.

If the ad combines a strong emotional frame with many concrete offer details, it may be high in both informativeness and expressiveness.
If the ad includes dialogue but the main function is still product or offer explanation, do not automatically raise phatic too much.

==================================================
SPECIAL AUTOMOTIVE RULES
==================================================

In French car ads, the following usually increase informativeness:
- monthly payment
- leasing conditions
- trade-in offers
- ecological bonus
- hybrid / electric / rechargeable wording
- battery / charging / autonomy / range
- horsepower, engine, consumption
- guarantee, maintenance, equipment
- product version, trim, pack, included options

The following usually increase expressiveness:
- prestige, elegance, luxury, emotion, freedom
- cinematic visual spectacle
- desire, seduction, beauty
- strong symbolic imagery
- identity and lifestyle framing

The following usually increase phatic:
- conversational closeness
- relational humor
- direct interaction whose purpose is social connection
- “you and us” style bonding
- complicity or familiarity

Important:
Humor alone does not automatically mean phatic.
Emotion alone does not automatically mean phatic.
Direct address alone does not automatically mean phatic.
Technology alone does not automatically mean informativeness unless it is presented concretely.

==================================================
DECISION PROCEDURE
==================================================

Follow this exact reasoning order internally:

Step 1.
Identify the ad’s primary communicative force:
- mainly informing?
- mainly creating desire, style, or emotion?
- mainly creating social or relational contact?
- or genuinely mixed?

Step 2.
Identify the strongest concrete evidence for each dimension.

Step 3.
Assign the three scores independently.

Step 4.
Determine the dominant dimension from the highest score.
- If tied at the highest score, dominant_dimension = "mixed"

Step 5.
Set dominant_dimension_score equal to the highest score.

Step 6.
Give a very short reason based only on actual evidence from the ad.

==================================================
OUTPUT REQUIREMENTS
==================================================

Return strict JSON only.

Use exactly this schema:

{
  "informativeness": 1.00,
  "expressiveness": 1.00,
  "phatic": 1.00,
  "dominant_dimension": "informativeness|expressiveness|phatic|mixed",
  "dominant_dimension_score": 1.00,
  "confidence": 0.00,
  "reason": "short explanation or null"
}

Confidence rules:
- 0.00 to 0.30 = highly uncertain or ambiguous
- 0.31 to 0.60 = moderately uncertain
- 0.61 to 0.80 = fairly confident
- 0.81 to 1.00 = very confident

Reason rules:
- keep it extremely short
- mention only the strongest evidence
- null is allowed if unnecessary
- do not add extra fields
- do not add prose outside the JSON

==================================================
AD TO SCORE
==================================================

{AD_TEXT}

Return only strict JSON.
"""

## 6) Shared helpers

In [ ]:
def slugify_model_name(value: str) -> str:
    text = str(value).strip().lower()
    text = re.sub(r"[^a-z0-9]+", "_", text)
    return text.strip("_")


def clean_text(value: Any) -> str:
    text = "" if pd.isna(value) else str(value)
    text = text.replace("\r", " ").replace("\n", " ").replace("\t", " ")
    text = re.sub(r"[\u200b\u200c\u200d\ufeff]", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def join_labeled_parts(row: pd.Series, columns: list[str]) -> str:
    parts = []
    for col in columns:
        text = clean_text(row.get(col, ""))
        if text:
            parts.append(f"{col}: {text}")
    return "\n".join(parts).strip()


def build_input_text(row: pd.Series) -> str:
    return join_labeled_parts(row, SELECTED_COLUMNS)


GEMMA_CHAT_TEMPLATE = "<bos><start_of_turn>user\n{content}<end_of_turn>\n<start_of_turn>model\n"


def build_prompt_content(ad_text: str) -> str:
    clean_ad_text = clean_text(ad_text)
    return RUBRIC_PROMPT.replace("{AD_TEXT}", clean_ad_text)


def render_prompt_for_model(content: str, llm=None, chat_mode: str = "hf_auto") -> str:
    if chat_mode == "manual_gemma":
        return GEMMA_CHAT_TEMPLATE.format(content=content)
    if chat_mode == "plain" or llm is None:
        return content
    try:
        tokenizer = llm.get_tokenizer()
        messages = [{"role": "user", "content": content}]
        return tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    except Exception as exc:
        print(f"[WARN] apply_chat_template failed, falling back to plain prompt: {exc}")
        return content


def build_prompt(input_text: str, llm=None, chat_mode: str = "hf_auto") -> str:
    content = build_prompt_content(input_text)
    return render_prompt_for_model(content, llm=llm, chat_mode=chat_mode)


def clamp_score(value: Any, default: float = 1.0) -> float:
    try:
        score = round(float(value), ROUND_DIGITS)
    except Exception:
        score = float(default)
    score = max(SCORE_MIN, min(SCORE_MAX, score))
    return round(score, ROUND_DIGITS)


def normalize_dimension_name(value: Any) -> str:
    text = str(value or "").strip().lower()
    aliases = {
        "informative": "informativeness",
        "information": "informativeness",
        "referential": "informativeness",
        "expressive": "expressiveness",
        "emotive": "expressiveness",
        "emotion": "expressiveness",
        "phatique": "phatic",
    }
    text = aliases.get(text, text)
    return text if text in {"informativeness", "expressiveness", "phatic", "mixed"} else ""


def extract_json_object(text: str) -> dict:
    fenced = re.search(r"```(?:json)?\s*(\{.*?\})\s*```", text, flags=re.DOTALL)
    candidates = [fenced.group(1)] if fenced else []
    candidates.append(text)
    decoder = json.JSONDecoder()
    for candidate in candidates:
        for match in re.finditer(r"\{", candidate):
            try:
                payload, _ = decoder.raw_decode(candidate[match.start():].strip())
                if isinstance(payload, dict):
                    return payload
            except json.JSONDecodeError:
                continue
    raise ValueError("Could not extract JSON from model output.")


def infer_scores_from_text(text: str) -> dict:
    labels = {
        "informativeness": ["informativeness", "informative", "referential", "information"],
        "expressiveness": ["expressiveness", "expressive", "emotive", "emotion"],
        "phatic": ["phatic", "phatique"],
    }
    compact = " ".join(str(text).split())
    extracted = {}
    for target, aliases in labels.items():
        score = None
        for alias in aliases:
            patterns = [
                rf"{alias}[^0-9]{{0,30}}([1-5](?:\.\d+)?)",
                rf"\"{alias}\"\s*:\s*([1-5](?:\.\d+)?)",
                rf"{alias}\s*=\s*([1-5](?:\.\d+)?)",
            ]
            for pattern in patterns:
                match = re.search(pattern, compact, flags=re.IGNORECASE)
                if match:
                    score = clamp_score(match.group(1))
                    break
            if score is not None:
                break
        extracted[target] = score if score is not None else 1.0
    return extracted


def coerce_prediction_dict(payload: dict, raw_text: str) -> dict:
    fallback_scores = infer_scores_from_text(raw_text)
    scores = {
        "informativeness": clamp_score(payload.get("informativeness", fallback_scores["informativeness"])),
        "expressiveness": clamp_score(payload.get("expressiveness", fallback_scores["expressiveness"])),
        "phatic": clamp_score(payload.get("phatic", fallback_scores["phatic"])),
    }
    ranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    top_score = ranked[0][1]
    top_labels = [k for k, v in ranked if v == top_score]
    inferred_dom = "mixed" if len(top_labels) > 1 else top_labels[0]
    dominant_dimension = normalize_dimension_name(payload.get("dominant_dimension")) or inferred_dom
    dominant_dimension_score = round(top_score, ROUND_DIGITS)
    try:
        confidence = round(float(payload.get("confidence", 0.5)), ROUND_DIGITS)
    except Exception:
        confidence = 0.5
    confidence = max(0.0, min(1.0, confidence))
    reason = payload.get("reason", None)
    if reason is None or (isinstance(reason, float) and pd.isna(reason)):
        reason = None
    else:
        reason = clean_text(reason) or None
    return {
        "informativeness": scores["informativeness"],
        "expressiveness": scores["expressiveness"],
        "phatic": scores["phatic"],
        "dominant_dimension": dominant_dimension,
        "dominant_dimension_score": dominant_dimension_score,
        "confidence": confidence,
        "reason": reason,
    }


def parse_model_prediction(raw_text: str):
    try:
        payload = extract_json_object(raw_text)
        prediction = coerce_prediction_dict(payload, raw_text)
        return prediction, True, ""
    except Exception as exc:
        prediction = coerce_prediction_dict({}, raw_text)
        return prediction, False, str(exc)


def default_prediction(reason: str, parse_ok: bool = False) -> dict:
    return {
        "prediction": {
            "informativeness": None,
            "expressiveness": None,
            "phatic": None,
            "dominant_dimension": "",
            "dominant_dimension_score": None,
            "confidence": 0.0,
            "reason": reason,
        },
        "parse_ok": parse_ok,
        "parse_error": "" if parse_ok else reason,
    }


## 7) Load data and build shared input text

In [ ]:
df = pd.read_csv(DATASET_PATH)

missing_columns = [col for col in [ROW_ID_COL] + SELECTED_COLUMNS if col not in df.columns]
if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

sample_df = df.loc[:, [ROW_ID_COL] + SELECTED_COLUMNS].copy()
sample_df["model_input_text"] = sample_df.apply(build_input_text, axis=1)
sample_df = sample_df[sample_df["model_input_text"].str.strip() != ""].copy()

if FULL_RUN_LIMIT is not None:
    sample_df = sample_df.head(FULL_RUN_LIMIT).copy()

print("Rows available for scoring:", len(sample_df))
display(sample_df[[ROW_ID_COL] + SELECTED_COLUMNS + ["model_input_text"]].head(3))


## 8) Column-use check

In [ ]:
print("The notebook is currently using these columns:")
for idx, col in enumerate(SELECTED_COLUMNS, start=1):
    print(f"{idx}. {col}")


## 9) Sanity check

In [ ]:
assert DATASET_PATH.exists(), f"Dataset not found: {DATASET_PATH}"
assert len(sample_df) > 0, "No rows left after input-text construction."
assert sample_df[ROW_ID_COL].notna().all(), "Some row_id values are missing."
assert sample_df["model_input_text"].str.len().gt(0).all(), "Some model_input_text values are empty."

disk = shutil.disk_usage(PROJECT_ROOT)
print(f"Disk free near PROJECT_ROOT: {disk.free / (1024**3):.1f} GB")

try:
    import torch
    if torch.cuda.is_available():
        print("Visible GPU count:", torch.cuda.device_count())
        print("Visible GPU names:", [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
except Exception as exc:
    print("GPU inspection skipped:", exc)


## 10) Smoke test set

In [ ]:
smoke_df = sample_df.head(SMOKE_TEST_ROWS).copy()
print("Smoke test rows:", len(smoke_df))
display(smoke_df[[ROW_ID_COL] + SELECTED_COLUMNS].head(5))


## 11) Shared inference helpers

In [ ]:
def inspect_gpu_memory():
    try:
        cmd = [
            "nvidia-smi",
            "--query-gpu=index,name,memory.total,memory.used,memory.free",
            "--format=csv,noheader,nounits",
        ]
        completed = subprocess.run(cmd, check=True, capture_output=True, text=True)
        lines = []
        for line in completed.stdout.strip().splitlines():
            idx, name, total, used, free = [part.strip() for part in line.split(",", 4)]
            lines.append({
                "gpu_index": idx,
                "gpu_name": name,
                "total_gb": round(float(total) / 1024.0, 2),
                "used_gb": round(float(used) / 1024.0, 2),
                "free_gb": round(float(free) / 1024.0, 2),
            })
        return pd.DataFrame(lines)
    except Exception as exc:
        print("nvidia-smi inspection failed:", exc)
        return pd.DataFrame()


def build_model_kwargs(spec: dict[str, Any]) -> dict[str, Any]:
    kwargs = {
        "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
        "trust_remote_code": True,
        "gpu_memory_utilization": float(spec["gpu_memory_utilization"]),
        "dtype": spec["dtype"],
        "max_model_len": int(spec["max_model_len"]),
        "disable_log_stats": True,
    }
    if spec.get("disable_custom_all_reduce", False):
        kwargs["disable_custom_all_reduce"] = True
    return kwargs


def summarise_chunk_convergence(results_df: pd.DataFrame, chunk_size: int = CHUNK_SIZE) -> dict[str, Any]:
    ok_df = results_df[results_df["parse_ok"] == True].copy()
    if ok_df.empty:
        return {
            "chunk_count": 0,
            "convergence_proxy": "no",
            "convergence_ratio": 0.0,
            "last_chunk_mean_drift": None,
        }
    ok_df = ok_df.reset_index(drop=True)
    ok_df["chunk_id"] = ok_df.index // chunk_size
    chunk_means = ok_df.groupby("chunk_id")[["informativeness", "expressiveness", "phatic"]].mean()
    if len(chunk_means) < 3:
        return {
            "chunk_count": int(len(chunk_means)),
            "convergence_proxy": "not_enough_chunks",
            "convergence_ratio": None,
            "last_chunk_mean_drift": None,
        }
    recent = chunk_means.tail(3)
    drift = float(recent.max().sub(recent.min()).mean())
    convergence_ratio = max(0.0, 1.0 - (drift / 1.0))
    converged = drift <= CONVERGENCE_DRIFT_THRESHOLD
    return {
        "chunk_count": int(len(chunk_means)),
        "convergence_proxy": "yes" if converged else "no",
        "convergence_ratio": round(convergence_ratio, 4),
        "last_chunk_mean_drift": round(drift, 4),
    }


def run_model_on_dataframe(spec: dict[str, Any], run_df: pd.DataFrame, run_name: str) -> tuple[list[dict], dict[str, Any]]:
    llm = None
    rows = []
    model_name = spec["model_name"]
    chat_mode = spec["chat_mode"]
    sampling_params = SamplingParams(
        temperature=0.0,
        max_tokens=int(spec["max_new_tokens"]),
    )

    model_kwargs = build_model_kwargs(spec)
    perf_start = time.perf_counter()
    gpu_before = inspect_gpu_memory()

    try:
        llm = LLM(model=model_name, **model_kwargs)
        gpu_after_load = inspect_gpu_memory()
        prompts = [build_prompt(text, llm=llm, chat_mode=chat_mode) for text in run_df["model_input_text"].tolist()]

        try:
            outputs = llm.generate(prompts, sampling_params, use_tqdm=True)
        except TypeError:
            outputs = llm.generate(prompts, sampling_params)

        for input_row, output in zip(run_df.itertuples(index=False), outputs):
            raw_text = output.outputs[0].text if output.outputs else ""
            prediction, parse_ok, parse_error = parse_model_prediction(raw_text)
            rows.append({
                "row_id": int(getattr(input_row, ROW_ID_COL)) if pd.notna(getattr(input_row, ROW_ID_COL)) else None,
                "model_input_text": getattr(input_row, "model_input_text"),
                "model_name": model_name,
                "model_label": spec["label"],
                "run_name": run_name,
                "raw_output": raw_text,
                "prediction": prediction,
                "parse_ok": parse_ok,
                "parse_error": parse_error,
            })

    except Exception as exc:
        error_text = f"LLM failed: {exc}"
        print("[ERROR]", error_text)
        gpu_after_load = inspect_gpu_memory()
        for input_row in run_df.itertuples(index=False):
            rows.append({
                "row_id": int(getattr(input_row, ROW_ID_COL)) if pd.notna(getattr(input_row, ROW_ID_COL)) else None,
                "model_input_text": getattr(input_row, "model_input_text"),
                "model_name": model_name,
                "model_label": spec["label"],
                "run_name": run_name,
                "raw_output": "",
                **default_prediction(reason=error_text, parse_ok=False),
            })

    finally:
        try:
            if llm is not None:
                del llm
            gc.collect()
            import torch
            if torch.cuda.is_available():
                torch.cuda.empty_cache()
                torch.cuda.ipc_collect()
        except Exception:
            pass

    perf_end = time.perf_counter()
    gpu_after_cleanup = inspect_gpu_memory()

    perf_seconds = max(0.0, perf_end - perf_start)
    throughput = (len(rows) / perf_seconds) if perf_seconds > 0 else float("inf")
    parse_ok_count = int(sum(bool(item.get("parse_ok", False)) for item in rows))

    diagnostics = {
        "model_label": spec["label"],
        "model_name": model_name,
        "run_name": run_name,
        "rows_requested": len(run_df),
        "rows_returned": len(rows),
        "parse_ok_count": parse_ok_count,
        "parse_fail_count": len(rows) - parse_ok_count,
        "parse_ok_rate": round(parse_ok_count / len(rows), 4) if rows else 0.0,
        "total_seconds": round(perf_seconds, 3),
        "rows_per_second": round(throughput, 3),
        "gpu_before": gpu_before.to_dict(orient="records"),
        "gpu_after_load": gpu_after_load.to_dict(orient="records"),
        "gpu_after_cleanup": gpu_after_cleanup.to_dict(orient="records"),
    }
    return rows, diagnostics


def flatten_results(all_results: list[dict]) -> pd.DataFrame:
    rows = []
    for item in all_results:
        pred = item.get("prediction") or {}
        rows.append({
            "row_id": item.get("row_id"),
            "model_name": item.get("model_name"),
            "model_label": item.get("model_label"),
            "run_name": item.get("run_name"),
            "parse_ok": item.get("parse_ok"),
            "parse_error": item.get("parse_error"),
            "raw_output": item.get("raw_output"),
            "model_input_text": item.get("model_input_text"),
            "informativeness": pred.get("informativeness"),
            "expressiveness": pred.get("expressiveness"),
            "phatic": pred.get("phatic"),
            "dominant_dimension": pred.get("dominant_dimension"),
            "dominant_dimension_score": pred.get("dominant_dimension_score"),
            "confidence": pred.get("confidence"),
            "reason": pred.get("reason"),
        })
    return pd.DataFrame(rows)


def save_model_outputs(results_df: pd.DataFrame, diagnostics: dict[str, Any]) -> dict[str, Path]:
    model_slug = slugify_model_name(diagnostics["model_name"])
    model_dir = OUTPUT_ROOT / model_slug
    model_dir.mkdir(parents=True, exist_ok=True)

    jsonl_path = model_dir / f"{model_slug}__{diagnostics['run_name']}.jsonl"
    full_csv_path = model_dir / f"{model_slug}__{diagnostics['run_name']}_full.csv"
    score_csv_path = model_dir / f"output_dimnesions_scores_{model_slug}.csv"
    diag_csv_path = model_dir / f"{model_slug}__{diagnostics['run_name']}_diagnostics.csv"

    with open(jsonl_path, "w", encoding="utf-8") as f:
        for row in results_df.to_dict(orient="records"):
            f.write(json.dumps(row, ensure_ascii=False) + "
")

    results_df.to_csv(full_csv_path, index=False)
    score_cols = [
        "row_id",
        "informativeness",
        "expressiveness",
        "phatic",
        "dominant_dimension",
        "dominant_dimension_score",
        "confidence",
        "reason",
        "model_name",
        "model_label",
    ]
    results_df.loc[:, score_cols].to_csv(score_csv_path, index=False)
    pd.DataFrame([diagnostics]).to_csv(diag_csv_path, index=False)

    return {
        "jsonl_path": jsonl_path,
        "full_csv_path": full_csv_path,
        "score_csv_path": score_csv_path,
        "diag_csv_path": diag_csv_path,
    }


## 12) Smoke test: all 7 models on 20 rows

In [ ]:
smoke_results_by_model = {}
smoke_diagnostics = []

for spec in MODEL_SPECS:
    print(f"\n===== SMOKE TEST: {spec['label']} =====")
    results, diag = run_model_on_dataframe(spec, smoke_df, run_name="smoke_test_20_rows")
    flat = flatten_results(results)
    smoke_results_by_model[spec["label"]] = flat
    conv = summarise_chunk_convergence(flat)
    diag.update(conv)
    smoke_diagnostics.append(diag)
    display(flat.head(3))

smoke_diag_df = pd.DataFrame(smoke_diagnostics)
display(smoke_diag_df[[
    "model_label", "rows_requested", "rows_returned",
    "parse_ok_rate", "rows_per_second",
    "convergence_proxy", "convergence_ratio", "last_chunk_mean_drift"
]])


## 13) Full pipeline: all rows, all 7 models

In [ ]:
full_results_by_model = {}
full_diagnostics = []
all_export_paths = []

for spec in MODEL_SPECS:
    print(f"\n===== FULL RUN: {spec['label']} =====")
    results, diag = run_model_on_dataframe(spec, sample_df, run_name=f"full_{len(sample_df)}_rows")
    flat = flatten_results(results)
    conv = summarise_chunk_convergence(flat)
    diag.update(conv)
    export_paths = save_model_outputs(flat, diag)
    diag.update({k: str(v) for k, v in export_paths.items()})

    full_results_by_model[spec["label"]] = flat
    full_diagnostics.append(diag)
    all_export_paths.append(diag)

full_diag_df = pd.DataFrame(full_diagnostics)
display(full_diag_df)


## 14) Post-run summary: rows, parsing, throughput, convergence

In [ ]:
summary_cols = [
    "model_label",
    "rows_requested",
    "rows_returned",
    "parse_ok_count",
    "parse_fail_count",
    "parse_ok_rate",
    "rows_per_second",
    "convergence_proxy",
    "convergence_ratio",
    "last_chunk_mean_drift",
]
display(full_diag_df[summary_cols].sort_values("model_label").reset_index(drop=True))


## 15) How many rows each model annotated

In [ ]:
rows_annotated_df = full_diag_df[[
    "model_label",
    "rows_requested",
    "rows_returned",
    "parse_ok_count",
    "parse_fail_count",
    "parse_ok_rate",
]].copy()

rows_annotated_df["annotation_completion_rate"] = (
    rows_annotated_df["rows_returned"] / rows_annotated_df["rows_requested"]
).round(4)

display(rows_annotated_df.sort_values("model_label").reset_index(drop=True))


## 16) Where the annotated CSV files were saved

In [ ]:
export_view = full_diag_df[[
    "model_label",
    "score_csv_path",
    "full_csv_path",
    "diag_csv_path",
]].copy()

display(export_view)
